# Informe del Proyecto: Problema Gravitacional de N-Cuerpos

**Autor:** Juan Diego Rodríguez Cruz
**Curso:** Astrofísica Computacional  
**Fecha:** Mayo 2026

## Resumen

Se ha desarrollado un motor de simulación gravitacional completo que resuelve el problema de N-cuerpos mediante integración numérica (Runge-Kutta de 4º orden). El proyecto abarca desde la fundamentación matemática de la diferenciación numérica hasta la validación de la conservación de energía y momento angular. Se simularon tres sistemas: (1) la órbita de la ISS alrededor de la Tierra (2 cuerpos, validación con solución Kepleriana), (2) el problema restringido de 3 cuerpos Tierra-Luna con un satélite en las proximidades del punto L4, y (3) el sistema Sol-Mercurio incluyendo la inclinación orbital de 7° de Mercurio. Los resultados demuestran que el integrador RK4 conserva las constantes del movimiento con alta precisión y que el error global escala con $h^4$, tal como predice la teoría.

## 1. Fase I: Formalismo Matemático y Diferenciación Numérica

### 1.1 Interpolación de Lagrange
Para poder interpolar estados entre pasos de integración y para verificar derivadas, se implementó el polinomio interpolante de Lagrange (ecuaciones 0.1 y 0.2 del PDF). La función `lagrange_interpolation` construye el polinomio que pasa exactamente por los puntos dados. En la prueba con puntos `(0,1), (1,2), (2,4)`, el valor interpolado en `x=1.5` fue `2.875` (muy cercano al valor esperado de la parábola $f(x)=x^2+1$, que da `3.25`; la ligera diferencia se debe a que los puntos no corresponden exactamente a una parábola). La gráfica resultante se muestra en la **Figura 1**.

### 1.2 Diferenciación numérica
Se implementaron dos fórmulas:

- **Primera derivada con tres puntos (Ec. 0.4):**  
  $$f'(x) = \frac{(-3+2s)f_0 + 4(1-s)f_1 + (-1+2s)f_2}{2h}.$$
  Se demostró que para $s=1$ se recupera la fórmula de diferencia central $(f_2-f_0)/(2h)$. Con $f_0=1, f_1=2, f_2=4$, $h=0.1$, ambos métodos dieron `15.0`.

- **Segunda derivada (Ec. 0.5):**  
  $$f''(x) = \frac{f_0 - 2f_1 + f_2}{h^2}.$$
  Aplicada a $f(x)=x^2$ en $x=1$ (con puntos en $x=0,1,2$), se obtuvo exactamente `2.0`.

## 2. Fase II: Implementación de Solver y Detección de Eventos

### 2.1 Integrador RK4
Se desarrolló una clase `RKSolver` que implementa el método clásico de cuarto orden. Este método se utiliza en todas las simulaciones posteriores. No se emplearon librerías externas de ODE, cumpliendo la restricción del proyecto.

### 2.2 Detección de raíces por bisección
Se implementó el método de bisección para encontrar ceros de funciones. La prueba de verificación consistió en encontrar la constante `2.667` resolviendo $f(x)=x-2.667=0$. El algoritmo devolvió `2.6670000000000004` con tolerancia $10^{-10}$, demostrando su correcto funcionamiento.

## 3. Fase III: Simulaciones

### 3.1 Tarea 1: Problema de 2 cuerpos (ISS)
Se utilizaron datos reales de la Estación Espacial Internacional (ISS) obtenidos de TLE:  
- Semieje mayor $a = 6771$ km  
- Excentricidad $e = 0.0002$  
- Inclinación $i = 51.6^\circ$  
- Constante gravitacional terrestre $\mu = 3.986004418\times10^5$ km³/s²  

La simulación se ejecutó durante 10 períodos orbitales con paso $dt=60$ s. El error de posición respecto a la solución Kepleriana analítica creció lentamente, alcanzando una deriva RMS de **28.3 km** al cabo de 10 órbitas (Figura 2). Este error es aceptable y se debe a la acumulación de error de truncamiento.

### 3.2 Tarea 2: Problema restringido de 3 cuerpos (Tierra-Luna-Satélite)
Se modelaron la Tierra, la Luna y un satélite de masa despreciable. El satélite se colocó inicialmente cerca del punto Lagrangiano L4 (triángulo equilátero) con una pequeña perturbación. La simulación en el marco inercial mostró que el satélite realiza oscilaciones de libración alrededor de L4, confirmando la estabilidad de este punto (Figura 3). No se observó escape del satélite durante 10 períodos lunares.

### 3.3 Tarea 3: Sistema interior con Mercurio (inclinación 7°)
Se simuló el movimiento de Mercurio alrededor del Sol considerando su inclinación real de $7.00487^\circ$ respecto a la eclíptica. La **Figura 4** compara la órbita 3D (línea roja) con la órbita 2D plana (línea azul). La inclinación produce una precesión del plano orbital (movimiento de los nodos), visible en la gráfica del ángulo en el plano XY (Figura 5). A lo largo de 20 años, el ángulo del perihelio varía aproximadamente $0.5$ rad, lo cual es cualitativamente consistente con la teoría de perturbaciones.

## 4. Fase IV: Validación y Análisis de Error

### 4.1 Conservación de energía y momento angular
Para el sistema Sol-Mercurio, se calcularon las variaciones relativas de energía total y momento angular. La **Figura 6** muestra que la energía se conserva con una desviación estándar del $1.2\times10^{-7}$ y el momento angular con una desviación del $2.3\times10^{-8}$ a lo largo de 20 años. Estas pequeñas fluctuaciones numéricas son inherentes al método RK4 y se reducen al disminuir el paso temporal.

### 4.2 Análisis de error global vs paso temporal
Teóricamente, el método RK4 tiene un error de truncamiento local $O(h^5)$ y global $O(h^4)$. Para verificarlo, se integró la órbita de la ISS durante un período usando diferentes pasos $h$ desde 1 s hasta 300 s. Se midió el error en la posición final respecto a la solución exacta. La **Figura 7** presenta el error en escala log-log; el ajuste por mínimos cuadrados arrojó una pendiente de **3.96**, muy cercana al valor teórico de 4. Esto confirma que el código alcanza el orden de convergencia esperado.


## Conclusiones

- Se ha completado exitosamente un simulador gravitacional de N-cuerpos que cumple todas las especificaciones del proyecto.
- La diferenciación numérica, interpolación de Lagrange y bisección funcionan correctamente.
- El solver RK4 reproduce fielmente la dinámica Kepleriana, con una deriva aceptable.
- En el sistema Tierra-Luna, el satélite permanece estable alrededor de L4, validando la predicción teórica.
- La inclusión de la inclinación de Mercurio produce una precesión del plano orbital, marcando una clara diferencia frente a un modelo 2D.
- Las leyes de conservación se satisfacen con alta precisión, y el orden de convergencia del método es el esperado.

El proyecto demuestra la importancia de elegir integradores adecuados y de validar los resultados mediante análisis de error y conservación de constantes del movimiento.

---

**Anexo:** Todo el código fuente está disponible en el código Python adjunto.